# Milestone 2 Lab, notebook 3 of 6: A model drafts a transformation

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the business's records | Recall the question and the answer key, and explore the records profit is calculated from |
| 2. One calculation, two engines | Write a calculation by hand in pandas and in SQL, with test cases that judge both |
| **3. A model drafts a transformation** (you are here) | Ask a model to write the calculation, and let the tests judge it |
| 4. Which model? Accuracy, cost, and speed | Give the same tasks to several models and compare them |
| 5. An agent that tests and fixes its own code | Give a model the tests as a tool, and compare it with a single draft |
| 6. The pipeline, and the same question again | Run the calculations on each engine and ask the analyst the leader's question |

Model requests in this notebook: **3**. A full run of this lab costs well under a dollar.

## What this notebook is for

In milestone 1, AI answered the business question. Here AI does a different job: it **writes the calculation**.

This is where an AI data engineer spends much of their time. A model can draft a transformation in seconds. Your job is to know whether the draft is right, and you already built the means to know: the hand-worked test cases.

In [1]:

from lab_helpers import cases_table, connect, save_result, show_code, show_doc

from retail_profit_stack.transform.drafting import DraftImplementation, make_task_prompt, run_draft
from retail_profit_stack.transform.specs import SPECS, TASKS

client, MODEL = connect()

Connected to OpenRouter. Default model: anthropic/claude-haiku-5.5


## Step 1: Who does what

The split of work is the same as in milestone 1: the model does what it is good at, and code decides whether the result can be trusted.

In [2]:
show_doc("HLD.md", "7. Where AI is used and where it is not")

> **From `docs/HLD.md`: 7. Where AI is used and where it is not**

| Task | Done by |
|---|---|
| Drafting a transformation in SQL or pandas from its contract | AI model |
| Fixing a draft after seeing which test cases failed | AI agent, using a test-running tool |
| Deciding whether a draft is correct | Hand-worked test cases (code) |
| Comparing models on accuracy, cost, and speed | Code, from test results and usage figures |
| Writing the test cases and their expected outputs | A person, by hand |
| Choosing what goes into production | A person, after the tests pass |
| Calculating business figures | Reviewed, versioned implementations, run by an engine |
| Answering the leader's question | The analyst agent, as in milestone 1 |

The model never produces a business figure. It produces code, and the code is trusted only after tests pass and a person has reviewed it.

## Step 2: Turn the contract into a prompt

We don't write the prompt by hand. The contract already says everything a model needs: the rule, the input tables and their columns, and the exact output. `make_task_prompt` turns a contract into a prompt.

Here is the prompt for `sales` in SQL. Read it as the model will.

In [3]:
spec = SPECS["sales"]
prompt = make_task_prompt(spec, "sql")

print(prompt)

You are a data engineer at a retailer. Write the transformation "sales".

# The rule

For order lines whose order is not cancelled, grouped by the month of the order's order_date and by product_id: units_sold is the sum of quantity; gross_sales is the sum of quantity x unit_list_price; discounts is the sum of discount_amount. A line that appears more than once is counted each time.

# Input tables

Table orders:
  - order_id: string
  - order_date: date
  - status: string  (completed or cancelled)

Table order_items:
  - order_item_id: string
  - order_id: string
  - product_id: string
  - quantity: integer
  - unit_list_price: decimal  (list price of one unit)
  - discount_amount: decimal  (discount on the whole line)

# Output

Exactly these columns, with these names:
  - month: month
  - product_id: string
  - units_sold: integer
  - gross_sales: decimal
  - discounts: decimal

One row for each combination of month and product_id. A month is text like 2026-10.
Amounts are in dollars

## Step 3: Ask for a draft

This is the structured call from milestone 1. The form this time has two fields: a line or two of reasoning, and the code.

In [4]:
response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    response_format=DraftImplementation,
    extra_body={"provider": {"require_parameters": True}},
)
draft = response.choices[0].message.parsed

print("The model's reasoning:", draft.reasoning)
show_code(draft.code, "sql")

The model's reasoning: Join order_items to orders on order_id and keep lines whose status is not 'cancelled' (IS DISTINCT FROM so a null status is kept). Group by the formatted month and product_id, summing quantity, quantity times unit price, and discount_amount, with no deduplication so repeated lines count each time. units_sold is cast to INTEGER, and SQL's null-preserving SUM keeps missing values as null rather than zero.


```sql
SELECT strftime(o.order_date, '%Y-%m') AS month,
  i.product_id AS product_id,
  CAST(SUM(i.quantity) AS INTEGER) AS units_sold,
  SUM(i.quantity * i.unit_list_price) AS gross_sales,
  SUM(i.discount_amount) AS discounts
FROM order_items i
JOIN orders o ON o.order_id = i.order_id
WHERE o.status IS DISTINCT FROM 'cancelled'
GROUP BY 1, 2
```

It looks reasonable. As with every AI output so far: looking reasonable is not the test.

## Step 4: Run it, carefully

Before running it, stop and think about what this is. It is **code written by a model, about to run on your computer**. Most of the time it is harmless. But it is untrusted, and it should be treated that way.

The LLD sets limits on how a draft is run.

In [5]:
show_doc("LLD.md", "9. AI drafting")

> **From `docs/LLD.md`: 9. AI drafting**

**The task.** `make_task_prompt(spec, language, context)` builds the prompt from the spec: the rule, the input table columns, and the output columns. `language` is `sql` or `pandas`. `context` is `trimmed` (only the tables this transformation reads) or `full` (every table in the system, as background).

The prompt mentions how to handle dates only when one of the task's inputs has a date column. An earlier version gave that advice on every task, and several models misapplied it to tasks with no dates. Removing it lifted them from between 64% and 86% of cases passed to 93% or more.

**The reply.** A structured answer:

```python
class DraftImplementation(BaseModel):
    reasoning: str = Field(description="Two or three sentences on the approach")
    code: str = Field(description="Only the SQL query, or only the Python function")
```

**Running a draft.** A draft is untrusted code. `run_draft(code, language, spec)` applies these limits before and while running it against the test cases:

| Language | Limits |
|---|---|
| SQL | Must be a single `SELECT` statement. Runs in an in-memory DuckDB database with access to files and the network switched off. Time limit of 10 seconds |
| pandas | Checked before it runs: it must define `transform`, may import only `pandas` and `numpy`, and may not use file, network, process, or dynamic-execution calls. Then it runs in a separate process with a time limit of 10 seconds |

A model's reply is capped at 12,000 tokens, so a model that rambles cannot run on indefinitely. A draft that breaks a limit is scored as failing every case, with the reason recorded. These limits reduce risk; they are not a perfect sandbox, and the lab says so.

**Scoring a draft.** `DraftResult` records the model, language, transformation, cases passed out of total, the first failure message, input and output tokens, cost in dollars as reported by OpenRouter, and seconds taken.

`run_draft` applies those limits, runs the draft on every hand-worked case for the transformation, and reports what happened.

In [6]:
results = run_draft(draft.code, "sql", spec)

cases_table(results)

,case,passed,what went wrong
0,a plain order,yes,
1,a discounted line,yes,
2,a cancelled order is left out,yes,
3,two months and two products,yes,
4,a line that appears twice is counted twice,yes,


Each row is one hand-worked case. A capable model usually passes all five for `sales`, because it is a small, clearly stated task.

## Step 5: The simple approach: ask for everything at once

If a model can write one transformation, why not skip the six small steps and ask for the whole profit table in one go, straight from the raw records?

That task exists in the lab as `profit_all_in_one`. Its contract has a much longer rule.

In [7]:
all_in_one = TASKS["profit_all_in_one"]

print("Inputs:", all_in_one.inputs)
print()
print(all_in_one.rule)

Inputs: ('orders', 'order_items', 'refunds', 'shipments', 'payment_fees', 'product_costs')

Calculate monthly profit per product straight from the business's records. One row for every month and product that has sales. Sales count in the month of the order's order_date and leave out cancelled orders: units_sold is the sum of quantity, gross_sales the sum of quantity x unit_list_price, discounts the sum of discount_amount. refunds (sum of refund_amount) and return_cost (sum of return_handling_cost) count in the month of refund_date, whenever the order was placed. fulfillment_cost (sum of pick_pack_cost) and shipping_cost (sum of carrier_cost) count in the month of ship_date. payment_fees (sum of fee_amount) count in the month of capture_date. Where a product has none of these in a month, use 0.00. product_cost is units_sold x the unit_cost for that product and month. net_sales = gross_sales - discounts - refunds. gross_profit = net_sales - product_cost. contribution_profit = gross_profi

In [8]:
response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": make_task_prompt(all_in_one, "sql")}],
    response_format=DraftImplementation,
    extra_body={"provider": {"require_parameters": True}},
)
big_draft = response.choices[0].message.parsed

print("The model's reasoning:", big_draft.reasoning)
show_code(big_draft.code, "sql")

The model's reasoning: Each measure is aggregated separately by its own month key (order_date for sales, refund_date for refunds and return costs, ship_date for fulfillment and shipping, capture_date for fees) so that joins cannot multiply rows. Rows are keyed on the non-cancelled sales month and product, with missing components coalesced to 0.00, while product_cost and the profit figures stay NULL when no unit_cost exists for that month.


```sql
WITH sales AS (
  SELECT strftime(o.order_date, '%Y-%m') AS month, oi.product_id AS product_id, SUM(oi.quantity) AS units_sold, SUM(oi.quantity * oi.unit_list_price) AS gross_sales, SUM(oi.discount_amount) AS discounts
  FROM order_items oi
  JOIN orders o ON o.order_id = oi.order_id
  WHERE o.status <> 'cancelled'
  GROUP BY 1, 2
),
refund_agg AS (
  SELECT strftime(refund_date, '%Y-%m') AS month, product_id, SUM(refund_amount) AS refunds, SUM(return_handling_cost) AS return_cost
  FROM refunds
  GROUP BY 1, 2
),
ship_agg AS (
  SELECT strftime(ship_date, '%Y-%m') AS month, product_id, SUM(pick_pack_cost) AS fulfillment_cost, SUM(carrier_cost) AS shipping_cost
  FROM shipments
  GROUP BY 1, 2
),
fee_agg AS (
  SELECT strftime(capture_date, '%Y-%m') AS month, product_id, SUM(fee_amount) AS payment_fees
  FROM payment_fees
  GROUP BY 1, 2
),
base AS (
  SELECT s.month, s.product_id, s.units_sold, s.gross_sales, s.discounts,
    COALESCE(r.refunds, 0.00) AS refunds,
    COALESCE(r.return_cost, 0.00) AS return_cost,
    COALESCE(sh.fulfillment_cost, 0.00) AS fulfillment_cost,
    COALESCE(sh.shipping_cost, 0.00) AS shipping_cost,
    COALESCE(f.payment_fees, 0.00) AS payment_fees,
    s.gross_sales - s.discounts - COALESCE(r.refunds, 0.00) AS net_sales,
    s.units_sold * pc.unit_cost AS product_cost
  FROM sales s
  LEFT JOIN refund_agg r ON r.month = s.month AND r.product_id = s.product_id
  LEFT JOIN ship_agg sh ON sh.month = s.month AND sh.product_id = s.product_id
  LEFT JOIN fee_agg f ON f.month = s.month AND f.product_id = s.product_id
  LEFT JOIN product_costs pc ON pc.month = s.month AND pc.product_id = s.product_id
)
SELECT month, product_id, units_sold, gross_sales, discounts, refunds, net_sales, product_cost,
  net_sales - product_cost AS gross_profit,
  fulfillment_cost, shipping_cost, payment_fees, return_cost,
  net_sales - product_cost - fulfillment_cost - shipping_cost - payment_fees - return_cost AS contribution_profit,
  '1.0.0' AS metric_version
FROM base
```

That is a lot of SQL to check by eye. Could you say, from reading it, whether it handles a line with two refunds correctly? Or a product with no cost? This is why we don't check by eye.

In [9]:
big_results = run_draft(big_draft.code, "sql", all_in_one)

cases_table(big_results)

,case,passed,what went wrong
0,one plain order,yes,
1,everything at once,yes,
2,a line loaded twice,yes,


Look at the case called **"everything at once"**. It contains a September order refunded in October, a line with two refunds, a cancelled order, and a product with no cost. It exists to catch the join trap from notebook 2 and the missing-cost rule.

A capable model may well pass it. A weaker one often does not. Either way, you now know something you could not have known by reading the code.

## Step 6: The same task in pandas

The contract has no language in it, so the same task can be requested in pandas. Only the last part of the prompt changes.

In [10]:
response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": make_task_prompt(SPECS["product_cost"], "pandas")}],
    response_format=DraftImplementation,
    extra_body={"provider": {"require_parameters": True}},
)
pandas_draft = response.choices[0].message.parsed

show_code(pandas_draft.code, "pandas")
cases_table(run_draft(pandas_draft.code, "pandas", SPECS["product_cost"]))

```python
import pandas as pd

def transform(inputs: dict[str, pd.DataFrame]) -> pd.DataFrame:
    sales = inputs['sales'][['month', 'product_id', 'units_sold']].copy()
    costs = inputs['product_costs'][['product_id', 'month', 'unit_cost']].copy()
    costs['unit_cost'] = pd.to_numeric(costs['unit_cost'], errors='coerce')
    costs = costs.drop_duplicates(subset=['month', 'product_id'], keep='first')
    merged = sales.merge(costs, on=['month', 'product_id'], how='left')
    merged['product_cost'] = pd.to_numeric(merged['units_sold'], errors='coerce').astype(float) * merged['unit_cost'].astype(float)
    out = merged[['month', 'product_id', 'product_cost']].copy()
    out['product_cost'] = out['product_cost'].astype(float)
    return out.reset_index(drop=True)
```

,case,passed,what went wrong
0,cost present,yes,
1,cost missing,yes,
2,the cost for the right month is used,yes,


`product_cost` has a case called **"cost missing"**. The rule says the camp stove's row must be kept, with its cost left empty. A draft that drops the row, or fills in zero, fails that case.

## What we learned

- **The contract writes the prompt.** The same description that people and engines read is what the model reads.
- **A draft is untrusted code.** It runs under limits, and it is judged by tests, never by how it looks.
- **Small tasks are easier to get right, and far easier to check**, than one large task. That is the reason for six small transformations.
- **The tests are what make AI useful here.** Without them, a draft is a guess you would have to read line by line. With them, it is a candidate you can accept or reject in a second.

One model, three drafts. But which model should a data team use for this? A bigger one is more reliable and costs more. A smaller one is cheap and fast and sometimes wrong. That is a question you can answer with evidence.

**Next: notebook 4**, where several models get the same tasks.

In [11]:
save_result("first_drafts", {
    "model": MODEL,
    "sales_sql": {"code": draft.code, "passed": sum(result.passed for result in results), "total": len(results)},
    "all_in_one_sql": {"code": big_draft.code, "passed": sum(result.passed for result in big_results), "total": len(big_results)},
})

Saved results/first_drafts.json
